# Aula 2 — ETL do Brasileirão

**Missão:** extrair os jogos da Série A, descobrir os nomes das equipes, padronizar colunas e selecionar as partidas de um time em um ano. Depois, carregar o recorte no PostgreSQL.

A aula do Titanic continua em `bdia_titanic.ipynb`. Esta aula reutiliza o ambiente virtual e a conexão com o banco.

## 0. Ambiente e parâmetros

Selecione o Python de `.venv`. Se precisar instalar as bibliotecas, use `%pip install -r requirements.txt` em uma célula. Troque `TIME` e `ANO` abaixo para explorar outros recortes. O nome deve ser exatamente o publicado na fonte.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from brasileirao import (
    URL, COLUNAS, APAGAR, COLUNAS_DESEJADAS,
    extrair, listar_times, transformar, qualidade, carregar,
)
from pipeline import criar_engine

print(sys.executable)
TIME = "Palmeiras"
ANO = 2022

C:\Users\marcu\OneDrive\Documentos\titanic-etl\.venv\Scripts\python.exe


## 1. Extrair todos os jogos

`read_csv` recebe uma URL ou um caminho local e devolve um DataFrame. A extração não filtra as partidas. Preferimos a cópia local quando ela já existe; a fonte continua indicada em `URL`.

Não fixe o tamanho do dataset no código: ele pode ser atualizado pelo mantenedor.

In [2]:
print("Fonte:", URL)
cache = Path("data/brasileirao_bruto.csv")
origem = cache if cache.exists() else URL
dados_brutos = extrair(origem)
print("Linhas e colunas:", dados_brutos.shape)
dados_brutos.head()

Fonte: https://raw.githubusercontent.com/adaoduque/Brasileirao_Dataset/master/campeonato-brasileiro-full.csv
Chegaram 8785 jogos.
Linhas e colunas: (8785, 16)


,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado
0,1,1,29/03/2003,16:00,Guarani,Vasco,NaN,NaN,NaN,NaN,Guarani,Brinco de Ouro,4,2,SP,RJ
1,2,1,29/03/2003,16:00,Athletico-PR,Gremio,NaN,NaN,NaN,NaN,Athletico-PR,Arena da Baixada,2,0,PR,RS
2,3,1,30/03/2003,16:00,Flamengo,Coritiba,NaN,NaN,NaN,NaN,-,Maracanã,1,1,RJ,PR
3,4,1,30/03/2003,16:00,Goias,Paysandu,NaN,NaN,NaN,NaN,-,Serra Dourada,2,2,GO,PA
4,5,1,30/03/2003,16:00,Internacional,Ponte Preta,NaN,NaN,NaN,NaN,-,Beira Rio,1,1,RS,SP


In [3]:
dados_brutos.isna().sum().rename("ausentes").to_frame()

,ausentes
ID,0
rodata,0
data,0
hora,0
mandante,0
visitante,0
formacao_mandante,4975
formacao_visitante,4975
tecnico_mandante,4610
tecnico_visitante,4610


## 2. Descobrir times e anos

`unique()` retorna os valores distintos de uma coluna. Use os parênteses para chamar o método. `sorted()` ordena esses nomes.

A função `listar_times` combina mandantes e visitantes, incluindo uma equipe mesmo que ela apareça apenas como visitante.

In [4]:
print("Mandantes:", sorted(dados_brutos["mandante"].dropna().unique()))
print("Todas as equipes:", listar_times(dados_brutos))

Mandantes: ['America-MG', 'America-RN', 'Athletico-PR', 'Atletico-GO', 'Atletico-MG', 'Avai', 'Bahia', 'Barueri', 'Botafogo-RJ', 'Bragantino', 'Brasiliense', 'CSA', 'Ceara', 'Chapecoense', 'Corinthians', 'Coritiba', 'Criciuma', 'Cruzeiro', 'Cuiaba', 'Figueirense', 'Flamengo', 'Fluminense', 'Fortaleza', 'Goias', 'Gremio', 'Gremio Prudente', 'Guarani', 'Internacional', 'Ipatinga', 'Joinville', 'Juventude', 'Nautico', 'Palmeiras', 'Parana', 'Paysandu', 'Ponte Preta', 'Portuguesa', 'Santa Cruz', 'Santo Andre', 'Santos', 'Sao Caetano', 'Sao Paulo', 'Sport', 'Vasco', 'Vitoria']
Todas as equipes: ['America-MG', 'America-RN', 'Athletico-PR', 'Atletico-GO', 'Atletico-MG', 'Avai', 'Bahia', 'Barueri', 'Botafogo-RJ', 'Bragantino', 'Brasiliense', 'CSA', 'Ceara', 'Chapecoense', 'Corinthians', 'Coritiba', 'Criciuma', 'Cruzeiro', 'Cuiaba', 'Figueirense', 'Flamengo', 'Fluminense', 'Fortaleza', 'Goias', 'Gremio', 'Gremio Prudente', 'Guarani', 'Internacional', 'Ipatinga', 'Joinville', 'Juventude', 'Nauti

### Ano a partir da string

A fonte usa `data` (singular) em `DD/MM/AAAA`. O slice `.str[-4:]` seleciona os quatro últimos caracteres, e `.astype(int)` converte o texto em inteiro. A sintaxe é `astype(int)`, com parênteses.

Essa técnica pressupõe datas nesse formato. No pipeline reutilizável, convertemos a coluna com `pd.to_datetime(..., format="%d/%m/%Y")`, que também detecta datas inválidas; depois usamos `.dt.year`.

In [5]:
ano_por_slice = dados_brutos["data"].str[-4:].astype(int)
datas = pd.to_datetime(dados_brutos["data"], format="%d/%m/%Y", errors="raise")
pd.testing.assert_series_equal(ano_por_slice, datas.dt.year, check_dtype=False)
print("Anos disponíveis:", sorted(ano_por_slice.unique()))

Anos disponíveis: [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


**Ano civil não é temporada.** Este exercício filtra pelo ano em que o jogo ocorreu. Partidas da temporada 2020 disputadas em 2021 entram no filtro de 2021. A fonte não fornece uma coluna de temporada.

Não imponha 38 como limite geral: os campeonatos de 2003 a 2005 tinham outro formato. Para Palmeiras em 2022, esperamos observar 38 jogos na fonte atual.

## 3. Padronizar a estrutura

`rename(columns=COLUNAS)` troca cabeçalhos pelo dicionário. Corrigimos `rodata → rodada`, `ID → id_jogo`, `arena → estadio` e padronizamos gols e estados.

Removemos as quatro colunas de técnico e formação pedidas no material. Os valores ausentes dessas colunas não precisam ser inventados.

In [6]:
print("Renomear:", COLUNAS)
print("Remover:", APAGAR)
preparados = dados_brutos.rename(columns=COLUNAS).drop(columns=APAGAR).copy()
preparados["data"] = datas
preparados["ano"] = preparados["data"].dt.year
preparados.head()

Renomear: {'ID': 'id_jogo', 'rodata': 'rodada', 'arena': 'estadio', 'mandante_Placar': 'gols_mandante', 'visitante_Placar': 'gols_visitante', 'mandante_Estado': 'estado_mandante', 'visitante_Estado': 'estado_visitante'}
Remover: ['formacao_mandante', 'formacao_visitante', 'tecnico_mandante', 'tecnico_visitante']


,id_jogo,rodada,data,hora,mandante,visitante,vencedor,estadio,gols_mandante,gols_visitante,estado_mandante,estado_visitante,ano
0,1,1,2003-03-29,16:00,Guarani,Vasco,Guarani,Brinco de Ouro,4,2,SP,RJ,2003
1,2,1,2003-03-29,16:00,Athletico-PR,Gremio,Athletico-PR,Arena da Baixada,2,0,PR,RS,2003
2,3,1,2003-03-30,16:00,Flamengo,Coritiba,-,Maracanã,1,1,RJ,PR,2003
3,4,1,2003-03-30,16:00,Goias,Paysandu,-,Serra Dourada,2,2,GO,PA,2003
4,5,1,2003-03-30,16:00,Internacional,Ponte Preta,-,Beira Rio,1,1,RS,SP,2003


## 4. Filtrar: OU entre os times, E com o ano

Uma partida pertence ao time se ele é mandante **ou** visitante: `|`. Ela também precisa ser do ano escolhido: `&`.

Coloque cada comparação entre parênteses. Use `==` para comparar; `=` faz uma atribuição. Em filtros de colunas do pandas usamos `|` e `&`.

In [7]:
filtro_do_time = (
    (preparados["mandante"] == TIME)
    | (preparados["visitante"] == TIME)
)
filtro_do_ano = preparados["ano"] == ANO

recorte_didatico = (
    preparados.loc[filtro_do_time & filtro_do_ano, COLUNAS_DESEJADAS]
    .sort_values(["data", "id_jogo"])
    .reset_index(drop=True)
)
print(f"{TIME}, {ANO}: {len(recorte_didatico)} jogos")
recorte_didatico.head()

Palmeiras, 2022: 38 jogos


,id_jogo,rodada,data,hora,ano,mandante,visitante,gols_mandante,gols_visitante,vencedor,estadio,estado_mandante,estado_visitante
0,7648,1,2022-04-09,21:00,2022,Palmeiras,Ceara,2,3,Ceara,Allianz Parque,SP,CE
1,7656,2,2022-04-16,16:30,2022,Goias,Palmeiras,1,1,-,Estádio da Serrinha,GO,SP
2,7666,4,2022-04-20,19:30,2022,Flamengo,Palmeiras,0,0,-,Maracanã,RJ,SP
3,7678,3,2022-04-23,19:00,2022,Palmeiras,Corinthians,3,0,Palmeiras,Arena Barueri,SP,SP
4,7689,5,2022-05-08,16:00,2022,Palmeiras,Fluminense,1,1,-,Allianz Parque,SP,RJ


## 5. Reutilizar a função transformar

Em `brasileirao.py`, `transformar(dados, time_desejado, ano_desejado)` reúne os passos anteriores, valida entradas e preserva o DataFrame bruto. Podemos repetir o processo trocando apenas os parâmetros.

In [8]:
dados_limpos = transformar(dados_brutos, TIME, ANO)
pd.testing.assert_frame_equal(dados_limpos, recorte_didatico)
assert len(dados_limpos) == dados_limpos["id_jogo"].nunique()
assert dados_limpos["ano"].eq(ANO).all()
assert ((dados_limpos["mandante"] == TIME) | (dados_limpos["visitante"] == TIME)).all()
qualidade(dados_limpos)

{'linhas': 38,
 'colunas': 13,
 'ids_duplicados': 0,
 'ausentes_por_coluna': {'id_jogo': 0,
  'rodada': 0,
  'data': 0,
  'hora': 0,
  'ano': 0,
  'mandante': 0,
  'visitante': 0,
  'gols_mandante': 0,
  'gols_visitante': 0,
  'vencedor': 0,
  'estadio': 0,
  'estado_mandante': 0,
  'estado_visitante': 0}}

In [9]:
destino = Path("data")
destino.mkdir(exist_ok=True)
dados_brutos.to_csv(destino / "brasileirao_bruto.csv", index=False)
dados_limpos.to_csv(destino / "brasileirao_filtrado.csv", index=False, date_format="%Y-%m-%d")
print("Arquivos salvos na pasta data.")

Arquivos salvos na pasta data.


## 6. Carregar no PostgreSQL

Configure `.env` e inicie o servidor conforme o README. A tabela desta aula é `jogos_brasileirao`; o Titanic permanece em `passageiros`.

A carga grava as partidas do recorte escolhido e preserva as de recortes anteriores. A chave primária `id_jogo` permite atualizar partidas já gravadas sem duplicá-las. Um recorte vazio é recusado. Use `substituir=True` para manter somente o novo recorte na tabela.

A carga fica desativada abaixo para permitir estudar a transformação sem um servidor ativo.

In [10]:
EXECUTAR_CARGA = False

if EXECUTAR_CARGA:
    engine = criar_engine()
    try:
        carregar(dados_limpos, engine)
    finally:
        engine.dispose()
else:
    print("Carga não executada. Configure o PostgreSQL e ative EXECUTAR_CARGA.")

Carga não executada. Configure o PostgreSQL e ative EXECUTAR_CARGA.


## Exercícios

1. Liste os times e troque Palmeiras por outro nome existente.
2. Mostre apenas os jogos como mandante; depois, apenas como visitante.
3. Conte vitórias, empates e derrotas comparando os gols com a posição do seu time.
4. Compare o ano civil 2020 com 2021. Por que uma equipe pode ter mais de 38 partidas em um ano?
5. Explique por que `.str[-4:]` depende do formato original da data.

Fontes: [dataset do Brasileirão](https://github.com/adaoduque/Brasileirao_Dataset), [CSV original](https://raw.githubusercontent.com/adaoduque/Brasileirao_Dataset/master/campeonato-brasileiro-full.csv), [pandas: filtros booleanos](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/indexing.html#boolean-indexing), [pandas.to_datetime](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.to_datetime.html).